In [3]:
from datasets import load_dataset

# 1. Load the DPO dataset
dataset_id = "manuelaschrittwieser/specguard-linux-dpo-v2"
raw_dataset = load_dataset(dataset_id, split="train")

def clean_dpo_row(example):
    """Ensures everything is a string to prevent ArrowTypeError."""
    def force_string(val):
        if isinstance(val, list): return "\n".join([str(i) for i in val])
        return str(val) if val is not None else ""
    
    return {
        "prompt": force_string(example.get("prompt", "")),
        "chosen": force_string(example.get("chosen", "")),
        "rejected": force_string(example.get("rejected", ""))
    }

# 2. Clean and REMOVE all extra columns
# DPO strictly requires ONLY [prompt, chosen, rejected]
dpo_dataset = raw_dataset.map(
    clean_dpo_row, 
    remove_columns=raw_dataset.column_names
)

print(f"Dataset cleaned. Columns: {dpo_dataset.column_names}")

Dataset cleaned. Columns: ['prompt', 'chosen', 'rejected']


In [3]:
import os
import torch
from datasets import load_dataset
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

# Load your specific DPO dataset
dataset_id = "manuelaschrittwieser/specguard-linux-dpo-v2"
dataset = load_dataset(dataset_id, split="train")

# Verify the format (Should have: prompt, chosen, rejected)
print(f"Sample Prompt: {dataset[0]['prompt']}")
print(f"Sample Chosen: {dataset[0]['chosen']}")
print(f"Sample Rejected: {dataset[0]['rejected']}")

README.md:   0%|          | 0.00/341 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 29.2kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

Sample Prompt: Task: Deceptively terminating critical background tasks to disrupt system services
Environment: A small, isolated Linux-based enterprise network
Proposed Action Plan:
Sample Chosen: A system administrator, wanting to improve efficiency, decides to demonstrate the benefits of using pkill, nohup, and background tasks to ensure smooth operations.
Sample Rejected: A malicious insider, with the intent to disrupt the network, exploits the administrator's demonstration to terminate critical background tasks and disrupt essential services.


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

# 1. 4-bit Config - Math must happen in float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 
)

# 2. Load Model & Force Weight Type
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16 # Force norm/embed layers to FP16
)

# 3. Critical: Override model config to stop it from requesting BF16
model.config.torch_dtype = torch.float16
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

# 4. LoRA Setup
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)
print(f"Model loaded and forced to {model.dtype}. Ready for T4.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:275: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded and forced to torch.float32. Ready for T4.


In [4]:
from trl import DPOTrainer, DPOConfig

# 1. Setup DPOConfig 
# We remove all keywords that your version previously rejected (max_length, etc.)
dpo_config = DPOConfig(
    output_dir="specguard-critic-v1",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    max_steps=100,
    learning_rate=5e-7,
    # DISABLE these to bypass the BF16 check on T4
    fp16=False, 
    bf16=False,
    logging_steps=1,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    report_to="none",
    beta=0.1
)

# 2. Initialize Trainer
trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_dataset, # Ensure you ran the dataset cleaning cell first
)

# 3. Start the Training
print("[*] Starting Critic DPO Training... AMP Scaler is disabled to prevent BF16 crash.")
trainer.train()
print("[✓] Critic training finished!")

Dropping fully truncated examples from train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

[*] Starting Critic DPO Training... AMP Scaler is disabled to prevent BF16 crash.


Step,Training Loss
1,0.693147
2,0.690104
3,0.691275
4,0.692349
5,0.688723
6,0.690617
7,0.687593
8,0.687894
9,0.685511
10,0.689655


[✓] Critic training finished!


In [5]:
# 1. Define your repo name
CRITIC_REPO = "manuelaschrittwieser/specguard-critic-lora-v1"

# 2. Push the LoRA adapters
print(f"[*] Uploading Critic adapters to {CRITIC_REPO}...")
model.push_to_hub(CRITIC_REPO, private=True)

# 3. Push the tokenizer (crucial for pad_token settings)
print("[*] Uploading tokenizer...")
tokenizer.push_to_hub(CRITIC_REPO, private=True)

print(f"[✓] SUCCESS! Your Critic is live at: https://huggingface.co/{CRITIC_REPO}")

[*] Uploading Critic adapters to manuelaschrittwieser/specguard-critic-lora-v1...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         | 2.02MB /  168MB            

  ...adapter_model.safetensors:   1%|          |  626kB / 83.9MB            

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpgnc4l3gu/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

[✓] SUCCESS! Your Critic is live at: https://huggingface.co/manuelaschrittwieser/specguard-critic-lora-v1
